# Data preparation and exploration

## Setup and imports

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../'

In [ ]:
%pip install -q tableone

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from tableone import TableOne

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1)

## Data import

In [ ]:
heart_disease = pd.read_csv(f'{PROJECT_ROOT}/data/heart_disease_cleveland_hungary.csv')

# Remove duplicates and null values, as per Straw et al.

rows_to_drop  = (heart_disease['ST slope'] == 0) | (heart_disease['cholesterol'] == 0) | (heart_disease['resting bp s'] == 0) | (heart_disease.duplicated(keep='first'))
heart_disease.drop(heart_disease[rows_to_drop].index, inplace=True)

# Clip negative values of st (oldpeak)
heart_disease['oldpeak'] = heart_disease['oldpeak'].clip(lower=0)

# print(heart_disease.describe())

## Study population summary

In [ ]:
from tableone import TableOne

# Descriptive statistics
table1 = TableOne(heart_disease,
                  groupby='sex',
                  continuous=['age','cholesterol','max heart rate','resting bp s','oldpeak'],
                  categorical=['target','chest pain type', 'fasting blood sugar','resting ecg','exercise angina','ST slope'],
                  missing=False
                  )

print(table1)

In [ ]:

heart_disease.rename(columns={'sex':'sex', 'chest pain type':'cp', 'resting bp s':'bp', 'cholesterol':'chol',
                              'fasting blood sugar':'fbs', 'resting ecg':'ecg', 'max heart rate':'mhr', 'exercise angina':'ang',
                              'oldpeak':'st', 'ST slope':'slope', 'target':'cvd'}, inplace=True)



## Data exploration

### Distributions

In [ ]:
# BP distribution
plt.figure(figsize=(10, 4))
sns.histplot(heart_disease, x='bp', bins=50, kde=True, stat='probability')
plt.title('Probability distribution of Resting Blood Pressure (BP)')
plt.show()

In [ ]:
# Chol distribution
plt.figure(figsize=(10, 4))
sns.histplot(heart_disease, x='chol', bins=50, kde=True, stat='probability')
plt.title('Probability distribution of Cholesterol (Chol)')
plt.show()

In [ ]:
# MHR distribution
plt.figure(figsize=(10, 4))
sns.histplot(heart_disease, x='mhr', bins=50, kde=True, stat='probability')
plt.title('Probability distribution of Max Heart Rate (MHR)')
plt.show()

In [ ]:
# Age distribution
plt.figure(figsize=(10, 4))
sns.histplot(heart_disease, x='age', bins=50, kde=True, stat='probability')
plt.title('Probability distribution of Age')
plt.show()

### Stratified distributions

In [ ]:
def format_mean_std(col):
  return f'{col.mean():.2f} ± {col.std():.2f}'

def format_median(col):
  return f'{col.median():.2f}'

def stratified_stats(df, var):
  df = heart_disease.pivot_table(index=['cvd','sex'],
                                 values=var, aggfunc=[format_mean_std, format_median])
  df.columns = df.columns.droplevel()
  df.columns = ['Mean +- Std', 'Median']
  return df

In [ ]:
g = sns.FacetGrid(heart_disease, col="cvd", hue='sex')
g.map(sns.kdeplot, 'bp')
g.add_legend()
plt.show()

print('\n')
print(stratified_stats(heart_disease, 'bp'))

In [ ]:
g = sns.FacetGrid(heart_disease, col="cvd", hue='sex')
g.map(sns.kdeplot, 'mhr')
g.add_legend()
plt.show()

print('\n')
print(stratified_stats(heart_disease, 'mhr'))

In [ ]:
g = sns.FacetGrid(heart_disease, col="cvd", hue='sex')
g.map(sns.kdeplot, 'chol')
g.add_legend()
plt.show()

print('\n')
print(stratified_stats(heart_disease, 'chol'))

In [ ]:
g = sns.catplot(data=heart_disease, x='ecg', col='cvd', kind='count', stat='proportion', hue='sex')
g.set_xticklabels(['normal','ST-T wave abnormality','probable LV hypertrophy'], rotation=45)
g.set_xlabels('')
plt.show()

## Pre-processing

In [ ]:
heart_disease_processed = heart_disease.copy()

# Z-score for age and mhr
cont_variables = ['mhr', 'age']
for var in cont_variables:
  heart_disease_processed[var] = (heart_disease_processed[var] - heart_disease_processed[var].mean()) / heart_disease_processed[var].std()

# Log and scaling for BP
log_chol = np.log(heart_disease_processed['chol'])
heart_disease_processed['chol'] = (log_chol - log_chol.mean()) / log_chol.std()

# Log and scaling for BP
log_bp = np.log(heart_disease_processed['bp'])
heart_disease_processed['bp'] = (log_bp - log_bp.mean()) / log_bp.std()

# Indexing cp and slope at 0
heart_disease_processed['cp'] = heart_disease_processed['cp'] - 1
heart_disease_processed['slope'] = heart_disease_processed['slope'] - 1

heart_disease_processed.reset_index(drop=True, inplace=True)

heart_disease_processed.to_csv(f'{PROJECT_ROOT}/data/heart_disease_cleaned.csv', index=False)